# Preprocesamiento y partición

## Entorno

Importé las librerías y fijé la semilla 42, que usan la partición y todos los modelos. Sin semilla,
K-Means y GMM arrancan desde centroides o componentes distintos en cada corrida, así que los grupos, su
numeración y las métricas cambiarían, y nadie podría verificar las cifras del informe. También imprimí
las versiones de Python, NumPy, pandas, scikit-learn y Matplotlib, porque los valores por defecto de los
estimadores cambian de una versión a otra y con ellos los resultados.

Definí tres funciones auxiliares. La función redondear aplica redondeo mitad hacia arriba una sola vez
sobre el valor completo. La usé en todas las cifras que se guardan porque round de Python trabaja sobre
la representación binaria y, por ejemplo, redondea 0,2175 a 0,217, mientras que el informe diría 0,218.
Así cada número se ve igual en el notebook y en el informe. La función guardar escribe la figura en
splits antes de mostrarla, porque plt.show libera la figura y el archivo saldría en blanco. La función
guardar_resultados agrega las cifras de cada notebook a resultados.json, de donde las toma el informe sin
copiarlas a mano.

In [1]:
import os, time, json, platform, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import joblib
from decimal import Decimal, ROUND_HALF_UP

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.dpi'] = 100
SEMILLA = 42
np.random.seed(SEMILLA)

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir('/content/drive/MyDrive'):
        raise RuntimeError('Drive no quedó montado. Vuelve a ejecutar esta celda y autoriza el acceso '
                           'en la ventana emergente.')
    RUTA = '/content/drive/MyDrive/credit_card'
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')

CARPETA_SPLITS = os.path.join(RUTA, 'splits')
os.makedirs(CARPETA_SPLITS, exist_ok=True)
ARCHIVO_RESULTADOS = os.path.join(RUTA, 'resultados.json')
resultados = {}
if os.path.exists(ARCHIVO_RESULTADOS):
    with open(ARCHIVO_RESULTADOS, encoding='utf-8') as f:
        resultados = json.load(f)

def redondear(valor, decimales=3):
    r = Decimal(repr(float(valor))).quantize(Decimal(1).scaleb(-decimales), rounding=ROUND_HALF_UP)
    return int(r) if decimales == 0 else float(r)

def redondear_serie(serie, decimales=3):
    return serie.map(lambda v: redondear(v, decimales))

def guardar(nombre):
    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_SPLITS, nombre + '.png'), dpi=150, bbox_inches='tight', facecolor='white')
    print('Gráfico guardado:', nombre + '.png')
    plt.show()

def guardar_resultados():
    with open(ARCHIVO_RESULTADOS, 'w', encoding='utf-8') as f:
        json.dump(resultados, f, ensure_ascii=False, indent=1,
                  default=lambda o: o.item() if hasattr(o, 'item') else str(o))
    print('Cifras guardadas en resultados.json')

resultados['versiones'] = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                           'scikit-learn': sklearn.__version__, 'matplotlib': matplotlib.__version__}
print('Guardando en Google Drive' if EN_DRIVE else 'Google Drive no disponible, guardando localmente')
print('Versiones:', resultados['versiones'])

Google Drive no disponible, guardando localmente
Versiones: {'python': '3.12.10', 'numpy': '2.2.6', 'pandas': '3.0.3', 'scikit-learn': '1.8.0', 'matplotlib': '3.10.9'}


## Carga de los datos

Leí la copia local CC_GENERAL.csv si ya existía y, si no, descargué el conjunto arjunbhasin2013/ccdata de
Kaggle con kagglehub, que no pide credenciales para datos públicos. Guardar la copia evita descargarlo en
cada notebook y asegura que los seis trabajen sobre el mismo archivo.

In [2]:
warnings.filterwarnings('ignore', message='IProgress not found')
import kagglehub

ARCHIVO_LOCAL = os.path.join(RUTA, 'CC_GENERAL.csv')

def cargar_datos():
    if os.path.exists(ARCHIVO_LOCAL):
        return pd.read_csv(ARCHIVO_LOCAL)
    carpeta = kagglehub.dataset_download('arjunbhasin2013/ccdata')
    datos = pd.read_csv(os.path.join(carpeta, 'CC GENERAL.csv'))
    datos.to_csv(ARCHIVO_LOCAL, index=False)
    return datos

datos = cargar_datos()
print('Filas:', datos.shape[0], '| Columnas:', datos.shape[1])

Filas: 8950 | Columnas: 18


## Corrección de rangos

Recorté a 1 las proporciones que lo superaban y quité CUST_ID. El recorte se aplica fila por fila y no
aprende nada de los datos, así que puede ir antes de la partición sin riesgo de fuga.

In [3]:
frecuencias = [c for c in datos.columns if 'FREQUENCY' in c or c == 'PRC_FULL_PAYMENT']
X = datos.drop(columns='CUST_ID').copy()
X[frecuencias] = X[frecuencias].clip(upper=1)
print('Proporciones mayores que 1 después del recorte:', int((X[frecuencias] > 1).sum().sum()))

Proporciones mayores que 1 después del recorte: 0


## Partición entrenamiento y prueba

Separé el 20 por ciento de los clientes como prueba, que hace el papel de los clientes nuevos que el
banco tendría que ubicar en un segmento. No estratifiqué porque no hay etiqueta. Desde aquí, todo lo que
aprende de los datos se ajusta solo con entrenamiento y después se aplica igual a prueba, como se
aplicaría a un cliente nuevo. Si el escalador o el PCA vieran a los clientes de prueba, las métricas de
prueba saldrían optimistas sin ningún síntoma visible.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(X, test_size=0.2, random_state=SEMILLA)
print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)
resultados['n_train'], resultados['n_test'] = len(X_train), len(X_test)

Entrenamiento: (7160, 17) | Prueba: (1790, 17)


Quedaron 7 160 clientes para entrenar y 1 790 para evaluar, los dos con las 17 variables.

## Transformación sin fuga

Armé un ColumnTransformer con dos ramas y lo ajusté con fit_transform sobre entrenamiento. A prueba solo
le apliqué transform, así que ninguna mediana, media ni desvío se calcula con clientes de prueba.

Los diez montos y conteos pasan por SimpleImputer con la mediana, por log1p y por StandardScaler. Imputé
con la mediana porque no se mueve con los atípicos de los montos y la media sí. FunctionTransformer con
np.log1p aplica log(1 + x) sin aprender nada. Sumar 1 permite transformar los ceros, y en CASH_ADVANCE el
51,7 por ciento de los clientes de entrenamiento tiene cero. La transformación comprime la cola derecha
que midió el notebook 02.

Las siete variables restantes (las seis frecuencias y proporciones y TENURE) solo pasan por la imputación
y por StandardScaler, que calcula z = (x − μ) / σ con la media y el desvío de entrenamiento. Escalar es
necesario porque K-Means y GMM trabajan con la distancia euclidiana, ||x − y||² = Σ_j (x_j − y_j)², donde
cada variable pesa en proporción a su varianza. Sin escalar, el límite de crédito (desvío de 3 662)
pesaría (3 662 / 0,20)² veces más que la frecuencia de avances en efectivo (desvío de 0,20).

El ColumnTransformer concatena las salidas en el orden de las ramas, primero los diez montos y después
las siete restantes. Por eso la lista COLUMNAS, que nombra las columnas de la matriz transformada en los
notebooks siguientes, es MONTOS seguida de RESTO.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer

MONTOS = ['BALANCE', 'PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE',
          'CASH_ADVANCE_TRX', 'PURCHASES_TRX', 'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS']
RESTO = [c for c in X.columns if c not in MONTOS]
COLUMNAS = MONTOS + RESTO

preprocesador = ColumnTransformer([
    ('montos', make_pipeline(SimpleImputer(strategy='median'), FunctionTransformer(np.log1p), StandardScaler()), MONTOS),
    ('resto', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), RESTO)])
X_tr = preprocesador.fit_transform(X_train)
X_te = preprocesador.transform(X_test)
print('Forma escalada:', X_tr.shape, X_te.shape)
print('Media y desvío en entrenamiento:', redondear(X_tr.mean(), 4), redondear(X_tr.std(), 4))
print('Media y desvío en prueba:', redondear(X_te.mean(), 4), redondear(X_te.std(), 4))
print('Nulos después de transformar:', int(np.isnan(X_tr).sum() + np.isnan(X_te).sum()))

Forma escalada: (7160, 17) (1790, 17)
Media y desvío en entrenamiento: -0.0 1.0
Media y desvío en prueba: 0.0081 0.9947
Nulos después de transformar: 0


En entrenamiento cada variable quedó con media 0 y desvío 1, como corresponde a un escalador ajustado con
esos datos. En prueba la media y el desvío quedaron cerca de esos valores sin ser exactos, que es lo
esperado cuando el escalador se aprende con otros clientes y se aplica sin reajustar. No quedó ningún
nulo.

## Marca de atípicos

Marqué a cada cliente de entrenamiento que tiene al menos un valor fuera de la regla del rango
intercuartílico del notebook 02. La marca no entra a ningún modelo. El notebook 04 la usa para ver en qué
grupos caen esos clientes y con qué certeza los asigna GMM.

In [6]:
q1, q3 = X_train.quantile(0.25), X_train.quantile(0.75)
iqr = q3 - q1
fuera = (X_train < q1 - 1.5 * iqr) | (X_train > q3 + 1.5 * iqr)
print('Clientes de entrenamiento con algún atípico:', int(fuera.any(axis=1).sum()))

Clientes de entrenamiento con algún atípico: 4795


## Datos preparados

Guardé en datos_preparados.joblib las dos particiones, en unidades originales y transformadas, junto con
el transformador, las listas de columnas y la marca de atípicos. Los notebooks 04, 05 y 06 cargan este
archivo en lugar de repetir el preprocesamiento.

In [7]:
joblib.dump({'X_train': X_train, 'X_test': X_test, 'X_tr': X_tr, 'X_te': X_te, 'preprocesador': preprocesador,
             'MONTOS': MONTOS, 'RESTO': RESTO, 'COLUMNAS': COLUMNAS, 'frecuencias': frecuencias, 'fuera': fuera},
            os.path.join(RUTA, 'datos_preparados.joblib'))
print('Datos preparados guardados en datos_preparados.joblib')
guardar_resultados()

Datos preparados guardados en datos_preparados.joblib
Cifras guardadas en resultados.json
